In [1]:
# ==============================================================================
# HỌC VIỆN CÔNG NGHỆ BƯU CHÍNH VIỄN THÔNG - KHOA CÔNG NGHỆ THÔNG TIN 1
# HỌC PHẦN: PHÁT TRIỂN HỆ THỐNG THÔNG MINH - ASSIGNMENT 01: DISCOVERING AI AGENT
# SINH VIÊN: NGUYỄN TUẤN TÀI | MSSV: B23DCCN731 | LỚP: 06 | NHÓM: 15
# ==============================================================================
import random
import math
import time
import json
from collections import deque
import numpy as np
import matplotlib.pyplot as plt

# Thiết lập hạt giống ngẫu nhiên để tái lập kết quả
random.seed(42)
np.random.seed(42)

print("[OK] Môi trường tính toán đã sẵn sàng.")
print("Hệ thống sẽ thực nghiệm 3 phần trọng tâm:")
print("  1. Cài đặt 5 lớp tác tử kinh điển (Russell & Norvig / IBM [2]): Simple Reflex, Model-based, Goal-based, Utility-based, Q-Learning RL.")
print("  2. Mô thức suy luận hiện đại ReAct Agent (Reasoning + Action) với Tool Calling.")
print("  3. Hệ thống Đa tác tử cộng tác (Multi-Agent System: Planner - Worker - Critic).")

[OK] Môi trường tính toán đã sẵn sàng.
Hệ thống sẽ thực nghiệm 3 phần trọng tâm:
  1. Cài đặt 5 lớp tác tử kinh điển (Russell & Norvig / IBM [2]): Simple Reflex, Model-based, Goal-based, Utility-based, Q-Learning RL.
  2. Mô thức suy luận hiện đại ReAct Agent (Reasoning + Action) với Tool Calling.
  3. Hệ thống Đa tác tử cộng tác (Multi-Agent System: Planner - Worker - Critic).


In [2]:
# ==============================================================================
# PHẦN 1.1: TÁC TỬ PHẢN XẠ ĐƠN GIẢN (SIMPLE REFLEX AGENT)
# Khái niệm (IBM [2] & Russell - Norvig): Ra quyết định dựa thuần túy trên tri giác hiện tại
# thông qua tập quy tắc Điều kiện - Hành động (Condition-Action Rules: IF/THEN).
# ==============================================================================

class SimpleReflexThermostatAgent:
    def __init__(self, target_temp=24.0):
        self.target_temp = target_temp
        
    def act(self, percept: dict) -> str:
        temp = percept.get("current_temp", 25.0)
        occupied = percept.get("is_occupied", True)
        
        # Tập luật phản xạ điều kiện - hành động (Condition-Action Rules)
        if not occupied:
            return "STANDBY_ECO (Phòng trống -> Tắt điều hòa tiết kiệm năng lượng)"
        elif temp < 20.0:
            return "HEAT_MAX (Nhiệt độ quá lạnh 18°C -> Bật sưởi tối đa công suất)"
        elif temp < self.target_temp - 1.0:
            return "HEAT_LOW (Hơi lạnh -> Bật sưởi nhẹ duy trì)"
        elif temp > 28.0:
            return "COOL_MAX (Nhiệt độ nóng 29°C -> Bật làm lạnh tối đa công suất)"
        elif temp > self.target_temp + 1.0:
            return "COOL_LOW (Hơi ấm -> Bật làm mát dịu)"
        else:
            return "HOLD_TEMP (Nhiệt độ lý tưởng 24°C -> Duy trì quạt gió nhẹ)"

# Thử nghiệm thực tế với chuỗi tri giác theo thời gian
agent_reflex = SimpleReflexThermostatAgent(target_temp=24.0)
environment_stream = [
    {"time": "08:00", "current_temp": 18.5, "is_occupied": True},
    {"time": "10:30", "current_temp": 22.0, "is_occupied": True},
    {"time": "12:00", "current_temp": 29.5, "is_occupied": True},
    {"time": "14:00", "current_temp": 24.2, "is_occupied": True},
    {"time": "18:00", "current_temp": 26.0, "is_occupied": False}
]

print("=== THỰC THI TÁC TỬ PHẢN XẠ ĐƠN GIẢN (SIMPLE REFLEX AGENT) ===")
for p in environment_stream:
    action = agent_reflex.act(p)
    print(f"[{p['time']}] Cảm biến: {p['current_temp']}°C, Có người: {p['is_occupied']} ==> Hành động: {action}")

=== THỰC THI TÁC TỬ PHẢN XẠ ĐƠN GIẢN (SIMPLE REFLEX AGENT) ===
[08:00] Cảm biến: 18.5°C, Có người: True ==> Hành động: HEAT_MAX (Nhiệt độ quá lạnh 18°C -> Bật sưởi tối đa công suất)
[10:30] Cảm biến: 22.0°C, Có người: True ==> Hành động: HEAT_LOW (Hơi lạnh -> Bật sưởi nhẹ duy trì)
[12:00] Cảm biến: 29.5°C, Có người: True ==> Hành động: COOL_MAX (Nhiệt độ nóng 29°C -> Bật làm lạnh tối đa công suất)
[14:00] Cảm biến: 24.2°C, Có người: True ==> Hành động: HOLD_TEMP (Nhiệt độ lý tưởng 24°C -> Duy trì quạt gió nhẹ)
[18:00] Cảm biến: 26.0°C, Có người: False ==> Hành động: STANDBY_ECO (Phòng trống -> Tắt điều hòa tiết kiệm năng lượng)


In [3]:
# ==============================================================================
# PHẦN 1.2: TÁC TỬ DỰA TRÊN MÔ HÌNH (MODEL-BASED REFLEX AGENT)
# Khái niệm (IBM [2]): Duy trì trạng thái nội tại (Internal State) để theo dõi các khía cạnh
# của thế giới mà cảm biến hiện tại không nhìn thấy trực tiếp (Partial Observability).
# ==============================================================================

class ModelBasedVacuumAgent:
    def __init__(self, rooms=["Phòng Khách", "Phòng Ngủ", "Bếp", "Phòng Làm Việc"]):
        self.rooms = rooms
        # Trạng thái nội tại (Internal State) lưu trữ bản đồ thế giới
        self.internal_state = {
            "current_location": "Phòng Khách",
            "room_dirt_status": {r: "DIRTY" for r in rooms},
            "cleaned_history": [],
            "battery_level": 100
        }
        
    def update_state(self, current_percept: dict):
        loc = current_percept["location"]
        dirt = current_percept["is_dirty"]
        self.internal_state["current_location"] = loc
        self.internal_state["room_dirt_status"][loc] = "DIRTY" if dirt else "CLEAN"
        self.internal_state["battery_level"] -= 5

    def choose_action(self) -> str:
        loc = self.internal_state["current_location"]
        status = self.internal_state["room_dirt_status"][loc]
        
        # Quyết định hành động dựa trên cả tri giác và mô hình nội tại
        if self.internal_state["battery_level"] <= 15:
            return "RETURN_TO_DOCK (Pin yếu -> Quay về trạm sạc)"
            
        if status == "DIRTY":
            self.internal_state["room_dirt_status"][loc] = "CLEAN"
            self.internal_state["cleaned_history"].append(loc)
            return f"SUCK_DIRT (Hút bụi dọn sạch tại {loc})"
        
        # Nếu phòng hiện tại đã sạch, tìm phòng bẩn tiếp theo trong mô hình bộ nhớ
        for r, s in self.internal_state["room_dirt_status"].items():
            if s == "DIRTY":
                return f"MOVE_TO (Di chuyển sang {r} còn bụi)"
                
        return "ALL_CLEAN (Tất cả các phòng trong bản đồ đã sạch -> Tự động nghỉ)"

# Mô phỏng quá trình dọn dẹp
agent_model = ModelBasedVacuumAgent()
percepts_log = [
    {"location": "Phòng Khách", "is_dirty": True},
    {"location": "Phòng Khách", "is_dirty": False},
    {"location": "Phòng Ngủ", "is_dirty": True},
    {"location": "Bếp", "is_dirty": True},
    {"location": "Phòng Làm Việc", "is_dirty": False}
]

print("=== THỰC THI TÁC TỬ DỰA TRÊN MÔ HÌNH (MODEL-BASED AGENT) ===")
for i, perc in enumerate(percepts_log, 1):
    agent_model.update_state(perc)
    act = agent_model.choose_action()
    print(f"Bước {i}: Vị trí = {perc['location']} | Bẩn = {perc['is_dirty']} | Pin = {agent_model.internal_state['battery_level']}%")
    print(f"  ↳ Hành động quyết định: {act}")
    print(f"  ↳ Bộ nhớ trạng thái nội tại: {agent_model.internal_state['room_dirt_status']}")

=== THỰC THI TÁC TỬ DỰA TRÊN MÔ HÌNH (MODEL-BASED AGENT) ===
Bước 1: Vị trí = Phòng Khách | Bẩn = True | Pin = 95%
  ↳ Hành động quyết định: SUCK_DIRT (Hút bụi dọn sạch tại Phòng Khách)
  ↳ Bộ nhớ trạng thái nội tại: {'Phòng Khách': 'CLEAN', 'Phòng Ngủ': 'DIRTY', 'Bếp': 'DIRTY', 'Phòng Làm Việc': 'DIRTY'}
Bước 2: Vị trí = Phòng Khách | Bẩn = False | Pin = 90%
  ↳ Hành động quyết định: MOVE_TO (Di chuyển sang Phòng Ngủ còn bụi)
  ↳ Bộ nhớ trạng thái nội tại: {'Phòng Khách': 'CLEAN', 'Phòng Ngủ': 'DIRTY', 'Bếp': 'DIRTY', 'Phòng Làm Việc': 'DIRTY'}
Bước 3: Vị trí = Phòng Ngủ | Bẩn = True | Pin = 85%
  ↳ Hành động quyết định: SUCK_DIRT (Hút bụi dọn sạch tại Phòng Ngủ)
  ↳ Bộ nhớ trạng thái nội tại: {'Phòng Khách': 'CLEAN', 'Phòng Ngủ': 'CLEAN', 'Bếp': 'DIRTY', 'Phòng Làm Việc': 'DIRTY'}
Bước 4: Vị trí = Bếp | Bẩn = True | Pin = 80%
  ↳ Hành động quyết định: SUCK_DIRT (Hút bụi dọn sạch tại Bếp)
  ↳ Bộ nhớ trạng thái nội tại: {'Phòng Khách': 'CLEAN', 'Phòng Ngủ': 'CLEAN', 'Bếp': 'CLEAN', 'Ph

In [4]:
# ==============================================================================
# PHẦN 1.3: TÁC TỬ DỰA TRÊN MỤC TIÊU (GOAL-BASED AGENT)
# Khái niệm (IBM [2]): Không chỉ phản xạ tức thời, tác tử sở hữu thông tin mục tiêu (Goal)
# và sử dụng thuật toán tìm kiếm/hoạch định (Search / Planning) để lập chuỗi hành động tới đích.
# ==============================================================================

class GoalBasedPathfinderAgent:
    def __init__(self, grid_map, start, goal):
        self.grid = grid_map # 0: Trống, 1: Chướng ngại vật
        self.start = start
        self.goal = goal
        self.height = len(grid_map)
        self.width = len(grid_map[0])
        
    def plan_path(self):
        """Thuật toán BFS lập kế hoạch tìm đường ngắn nhất tới Goal."""
        queue = deque([(self.start, [self.start])])
        visited = {self.start}
        
        while queue:
            (r, c), path = queue.popleft()
            if (r, c) == self.goal:
                return path # Đã đạt mục tiêu
                
            for dr, dc, move_name in [(-1, 0, 'UP'), (1, 0, 'DOWN'), (0, -1, 'LEFT'), (0, 1, 'RIGHT')]:
                nr, nc = r + dr, c + dc
                if 0 <= nr < self.height and 0 <= nc < self.width:
                    if self.grid[nr][nc] == 0 and (nr, nc) not in visited:
                        visited.add((nr, nc))
                        queue.append(((nr, nc), path + [(nr, nc)]))
        return None

# Bản đồ mê cung: S = (0,0), G = (4,4), 1 là vật cản
maze_grid = [
    [0, 0, 1, 0, 0],
    [1, 0, 1, 0, 1],
    [0, 0, 0, 0, 0],
    [0, 1, 1, 1, 0],
    [0, 0, 0, 0, 0]
]

start_pos = (0, 0)
goal_pos = (4, 4)
agent_goal = GoalBasedPathfinderAgent(maze_grid, start_pos, goal_pos)
optimal_path = agent_goal.plan_path()

print("=== THỰC THI TÁC TỬ DỰA TRÊN MỤC TIÊU (GOAL-BASED AGENT) ===")
print(f"Mục tiêu xuất phát: Start = {start_pos}  --->  Đích cần đến: Goal = {goal_pos}")
print(f"Kế hoạch hành động tối ưu tìm thấy ({len(optimal_path)-1} bước):")
for step, pos in enumerate(optimal_path):
    desc = "START" if step == 0 else ("GOAL_REACHED" if pos == goal_pos else "FORWARD")
    print(f"  Bước {step}: Di chuyển đến tọa độ {pos} [{desc}]")
print("[Thành công] Tác tử đã hoàn thành mục tiêu an toàn, tránh toàn bộ vật cản!")

=== THỰC THI TÁC TỬ DỰA TRÊN MỤC TIÊU (GOAL-BASED AGENT) ===
Mục tiêu xuất phát: Start = (0, 0)  --->  Đích cần đến: Goal = (4, 4)
Kế hoạch hành động tối ưu tìm thấy (8 bước):
  Bước 0: Di chuyển đến tọa độ (0, 0) [START]
  Bước 1: Di chuyển đến tọa độ (0, 1) [FORWARD]
  Bước 2: Di chuyển đến tọa độ (1, 1) [FORWARD]
  Bước 3: Di chuyển đến tọa độ (2, 1) [FORWARD]
  Bước 4: Di chuyển đến tọa độ (2, 2) [FORWARD]
  Bước 5: Di chuyển đến tọa độ (2, 3) [FORWARD]
  Bước 6: Di chuyển đến tọa độ (2, 4) [FORWARD]
  Bước 7: Di chuyển đến tọa độ (3, 4) [FORWARD]
  Bước 8: Di chuyển đến tọa độ (4, 4) [GOAL_REACHED]
[Thành công] Tác tử đã hoàn thành mục tiêu an toàn, tránh toàn bộ vật cản!


In [5]:
# ==============================================================================
# PHẦN 1.4: TÁC TỬ DỰA TRÊN ĐỘ THỎA DỤNG (UTILITY-BASED AGENT)
# Khái niệm (IBM [2]): Khi có nhiều cách đạt mục tiêu, tác tử sử dụng Hàm thỏa dụng (Utility Function)
# để định lượng mức độ hài lòng, cân bằng tối ưu giữa Phần thưởng, Chi phí và Rủi ro.
# ==============================================================================

class UtilityBasedDeliveryDroneAgent:
    def __init__(self, w_speed=0.4, w_cost=0.3, w_safety=0.3):
        # Trọng số ưu tiên của hàm thỏa dụng
        self.w_speed = w_speed
        self.w_cost = w_cost
        self.w_safety = w_safety
        
    def calculate_utility(self, route: dict) -> float:
        """
        Hàm Thỏa dụng: U(s) = w1 * Tốc độ - w2 * Chi phí pin + w3 * Độ an toàn thời tiết
        Chuẩn hóa các thành phần về thang điểm [0, 100].
        """
        speed_score = (60.0 / route["duration_mins"]) * 100.0
        cost_score = (1.0 - route["battery_cost_percent"] / 100.0) * 100.0
        safety_score = route["weather_safety_score"] # 0 - 100
        
        total_utility = (self.w_speed * speed_score + 
                         self.w_cost * cost_score + 
                         self.w_safety * safety_score)
        return round(total_utility, 2)
        
    def select_best_route(self, candidate_routes: list) -> tuple:
        best_route = None
        max_utility = -float('inf')
        
        for r in candidate_routes:
            u = self.calculate_utility(r)
            r["evaluated_utility"] = u
            if u > max_utility:
                max_utility = u
                best_route = r
        return best_route, candidate_routes

# 3 tuyến đường bay giao hàng ứng viên
routes = [
    {"name": "Tuyến A (Đường thẳng qua đồi)", "duration_mins": 15, "battery_cost_percent": 45, "weather_safety_score": 60},
    {"name": "Tuyến B (Đường vòng dọc bờ sông)", "duration_mins": 35, "battery_cost_percent": 25, "weather_safety_score": 95},
    {"name": "Tuyến C (Đường trung gian)", "duration_mins": 22, "battery_cost_percent": 30, "weather_safety_score": 85}
]

agent_drone = UtilityBasedDeliveryDroneAgent()
best_r, evaluated_list = agent_drone.select_best_route(routes)

print("=== THỰC THI TÁC TỬ DỰA TRÊN ĐỘ THỎA DỤNG (UTILITY-BASED AGENT) ===")
print("Đánh giá các tuyến đường bay dựa trên Hàm Thỏa Dụng U(s):")
for r in evaluated_list:
    print(f"• {r['name']}: Thời gian = {r['duration_mins']}p | Pin tốn = {r['battery_cost_percent']}% | An toàn = {r['weather_safety_score']}/100 ==> Điểm Thỏa Dụng = {r['evaluated_utility']}")

print(f"\n==> QUYẾT ĐỊNH TỐI ƯU CỦA AGENT: Lựa chọn '{best_r['name']}' với điểm thỏa dụng cao nhất ({best_r['evaluated_utility']}).")

=== THỰC THI TÁC TỬ DỰA TRÊN ĐỘ THỎA DỤNG (UTILITY-BASED AGENT) ===
Đánh giá các tuyến đường bay dựa trên Hàm Thỏa Dụng U(s):
• Tuyến A (Đường thẳng qua đồi): Thời gian = 15p | Pin tốn = 45% | An toàn = 60/100 ==> Điểm Thỏa Dụng = 194.5
• Tuyến B (Đường vòng dọc bờ sông): Thời gian = 35p | Pin tốn = 25% | An toàn = 95/100 ==> Điểm Thỏa Dụng = 119.57
• Tuyến C (Đường trung gian): Thời gian = 22p | Pin tốn = 30% | An toàn = 85/100 ==> Điểm Thỏa Dụng = 155.59

==> QUYẾT ĐỊNH TỐI ƯU CỦA AGENT: Lựa chọn 'Tuyến A (Đường thẳng qua đồi)' với điểm thỏa dụng cao nhất (194.5).


In [6]:
# ==============================================================================
# PHẦN 1.5: TÁC TỬ HỌC THÍCH NGHI (LEARNING AGENT VỚI HỌC TĂNG CƯỜNG - Q-LEARNING)
# Khái niệm (IBM [2] & GeeksforGeeks [4]): Tác tử tự học chính sách hành động thông qua
# tương tác với môi trường, nhận tín hiệu Thưởng/Phạt (Reward) để cập nhật ma trận Q-Table.
# Phương trình cập nhật Bellman: Q(s,a) = Q(s,a) + alpha * [R + gamma * max Q(s',a') - Q(s,a)]
# ==============================================================================

class QLearningGridAgent:
    def __init__(self, states=16, actions=4, alpha=0.1, gamma=0.9, epsilon=0.2):
        self.states = states
        self.actions = actions # 0: Lên, 1: Xuống, 2: Trái, 3: Phải
        self.alpha = alpha     # Tốc độ học (Learning rate)
        self.gamma = gamma     # Hệ số chiết khấu tương lai (Discount factor)
        self.epsilon = epsilon # Tỷ lệ khám phá ngẫu nhiên (Exploration rate)
        self.q_table = np.zeros((states, actions))
        
    def choose_action(self, state):
        if random.random() < self.epsilon:
            return random.randint(0, self.actions - 1) # Khám phá (Exploration)
        return np.argmax(self.q_table[state])          # Khai thác (Exploitation)
        
    def learn(self, state, action, reward, next_state):
        best_next = np.max(self.q_table[next_state])
        td_target = reward + self.gamma * best_next
        td_error = td_target - self.q_table[state, action]
        self.q_table[state, action] += self.alpha * td_error

# Môi trường 4x4 Grid (State 0 -> 15). Đích là State 15 (Reward = +100). Bẫy là State 5, 12 (Reward = -50).
episodes = 250
agent_rl = QLearningGridAgent()
reward_history = []

for ep in range(episodes):
    state = 0
    total_reward = 0
    steps = 0
    while state != 15 and steps < 30:
        action = agent_rl.choose_action(state)
        # Giả lập chuyển trạng thái
        if action == 1 and state + 4 < 16: next_state = state + 4 # Down
        elif action == 3 and (state + 1) % 4 != 0: next_state = state + 1 # Right
        elif action == 0 and state - 4 >= 0: next_state = state - 4 # Up
        elif action == 2 and state % 4 != 0: next_state = state - 1 # Left
        else: next_state = state
        
        # Phần thưởng môi trường
        if next_state == 15: reward = 100.0
        elif next_state in [5, 12]: reward = -50.0
        else: reward = -1.0 # Chi phí bước đi
        
        agent_rl.learn(state, action, reward, next_state)
        state = next_state
        total_reward += reward
        steps += 1
    reward_history.append(total_reward)

print("=== THỰC THI TÁC TỬ HỌC TĂNG CƯỜNG (Q-LEARNING AGENT - GEEKSFORGEEKS [4]) ===")
print(f"Số vòng huấn luyện (Episodes): {episodes}")
print(f"Phần thưởng tích lũy 10 tập đầu tiên: {np.mean(reward_history[:10]):.2f}")
print(f"Phần thưởng tích lũy 10 tập cuối cùng: {np.mean(reward_history[-10:]):.2f}")
print(f"Trạng thái hội tụ của Bảng Q-Table (Kích thước 16 trạng thái x 4 hành động):")
print(f"  Tại Trạng thái 14 (Sát Goal 15): Q[Phải] = {agent_rl.q_table[14, 3]:.2f} (Giá trị lớn nhất)")
print(f"  Tại Trạng thái 11 (Sát Goal 15): Q[Xuống] = {agent_rl.q_table[11, 1]:.2f} (Giá trị lớn nhất)")
print("[Kết luận] Tác tử đã tự học thành công chính sách tránh bẫy và đi tới đích với phần thưởng tối đa!")

=== THỰC THI TÁC TỬ HỌC TĂNG CƯỜNG (Q-LEARNING AGENT - GEEKSFORGEEKS [4]) ===
Số vòng huấn luyện (Episodes): 250
Phần thưởng tích lũy 10 tập đầu tiên: -39.60
Phần thưởng tích lũy 10 tập cuối cùng: 93.90
Trạng thái hội tụ của Bảng Q-Table (Kích thước 16 trạng thái x 4 hành động):
  Tại Trạng thái 14 (Sát Goal 15): Q[Phải] = 74.58 (Giá trị lớn nhất)
  Tại Trạng thái 11 (Sát Goal 15): Q[Xuống] = 100.00 (Giá trị lớn nhất)
[Kết luận] Tác tử đã tự học thành công chính sách tránh bẫy và đi tới đích với phần thưởng tối đa!


In [7]:
# ==============================================================================
# PHẦN 2: MÔ THỨC SUY LUẬN HIỆN ĐẠI REACT AGENT (REASONING AND ACTION)
# Khái niệm (IBM [2]): Kết hợp khả năng lập luận tư duy (Reasoning) và hành động (Action/Tool Calling)
# theo chu trình vòng lặp: Thought -> Action -> Observation -> Thought -> Final Answer.
# ==============================================================================

# Khởi tạo Hệ thống Công cụ Ngoại vi (Tools Ecosystem)
def tool_calculator(expr: str) -> str:
    """Công cụ máy tính an toàn tính toán biểu thức số học."""
    try:
        clean_expr = expr.replace('^', '**').strip()
        res = eval(clean_expr, {"__builtins__": None}, {})
        return f"{res}"
    except Exception as e:
        return f"Lỗi tính toán: {e}"

def tool_medical_db(query_drug: str) -> str:
    """Công cụ tra cứu cơ sở tri thức biệt dược và chống chỉ định y tế."""
    db = {
        "metformin": "Thuốc đầu tay điều trị Đái tháo đường Tuýp 2. Chống chỉ định: Suy thận nặng (eGFR < 30 mL/phút). Đơn giá Long Châu: 135.000 đ/hộp 50 viên.",
        "forxiga": "Thuốc ức chế SGLT-2, thải trừ đường qua thận và bảo vệ tim mạch. Đơn giá: 620.000 đ/hộp 28 viên.",
        "insulin": "Chỉ định Đái tháo đường Tuýp 1 hoặc Tuýp 2 thất bại thuốc uống. Bảo quản nhiệt độ 2-8°C."
    }
    key = query_drug.lower().strip()
    return db.get(key, "Không tìm thấy thông tin biệt dược trong cơ sở dữ liệu.")

class ReActAgent:
    def __init__(self):
        self.tools = {
            "calculator": tool_calculator,
            "medical_db": tool_medical_db
        }
        
    def run(self, user_prompt: str) -> str:
        trace = []
        trace.append(f"CÂU HỎI NGƯỜI DÙNG: {user_prompt}")
        
        # Bước 1: Suy luận (Thought 1)
        thought_1 = "Tôi cần tra cứu chống chỉ định của thuốc Metformin đối với bệnh nhân suy thận, sau đó tính chi phí dùng thuốc trong 60 ngày."
        trace.append(f"Thought 1: {thought_1}")
        
        # Bước 2: Hành động (Action 1 - Gọi Tool medical_db)
        action_1 = "medical_db[metformin]"
        trace.append(f"Action 1: {action_1}")
        obs_1 = self.tools["medical_db"]("metformin")
        trace.append(f"Observation 1: {obs_1}")
        
        # Bước 3: Suy luận (Thought 2)
        thought_2 = "Cơ sở dữ liệu nêu rõ Metformin chống chỉ định với suy thận nặng (eGFR < 30). Về chi phí: 1 hộp 50 viên giá 135.000đ (2.700đ/viên). Dùng 60 ngày x 2 viên/ngày = 120 viên. Tôi cần tính tổng tiền 120 * 2700."
        trace.append(f"Thought 2: {thought_2}")
        
        # Bước 4: Hành động (Action 2 - Gọi Tool calculator)
        action_2 = "calculator[120 * 2700]"
        trace.append(f"Action 2: {action_2}")
        obs_2 = self.tools["calculator"]("120 * 2700")
        trace.append(f"Observation 2: {obs_2} (VNĐ)")
        
        # Bước 5: Tổng hợp câu trả lời cuối cùng (Final Answer)
        final_answer = (
            f"1. Cảnh báo an toàn: Bệnh nhân suy thận nặng (eGFR < 30) CHỐNG CHỈ ĐỊNH tuyệt đối dùng Metformin do nguy cơ nhiễm toan lactic. Cần hỏi bác sĩ để chuyển sang nhóm thuốc an toàn cho thận (như DPP-4 Linagliptin).\n"
            f"2. Dự trù chi phí (nếu đủ điều kiện dùng 2 viên/ngày): Hết 120 viên trong 60 ngày, tổng chi phí là {int(float(obs_2)):,} VNĐ tại Nhà thuốc Long Châu."
        )
        trace.append(f"Final Answer:\n{final_answer}")
        return "\n".join(trace)

agent_react = ReActAgent()
output_trace = agent_react.run("Bệnh nhân suy thận có dùng được Metformin không và nếu uống 2 viên/ngày trong 60 ngày thì hết bao nhiêu tiền?")

print("=== THỰC THI MÔ THỨC SUY LUẬN REACT AGENT (REASONING + ACTION) ===")
print(output_trace)

=== THỰC THI MÔ THỨC SUY LUẬN REACT AGENT (REASONING + ACTION) ===
CÂU HỎI NGƯỜI DÙNG: Bệnh nhân suy thận có dùng được Metformin không và nếu uống 2 viên/ngày trong 60 ngày thì hết bao nhiêu tiền?
Thought 1: Tôi cần tra cứu chống chỉ định của thuốc Metformin đối với bệnh nhân suy thận, sau đó tính chi phí dùng thuốc trong 60 ngày.
Action 1: medical_db[metformin]
Observation 1: Thuốc đầu tay điều trị Đái tháo đường Tuýp 2. Chống chỉ định: Suy thận nặng (eGFR < 30 mL/phút). Đơn giá Long Châu: 135.000 đ/hộp 50 viên.
Thought 2: Cơ sở dữ liệu nêu rõ Metformin chống chỉ định với suy thận nặng (eGFR < 30). Về chi phí: 1 hộp 50 viên giá 135.000đ (2.700đ/viên). Dùng 60 ngày x 2 viên/ngày = 120 viên. Tôi cần tính tổng tiền 120 * 2700.
Action 2: calculator[120 * 2700]
Observation 2: 324000 (VNĐ)
Final Answer:
1. Cảnh báo an toàn: Bệnh nhân suy thận nặng (eGFR < 30) CHỐNG CHỈ ĐỊNH tuyệt đối dùng Metformin do nguy cơ nhiễm toan lactic. Cần hỏi bác sĩ để chuyển sang nhóm thuốc an toàn cho thận (như 

In [8]:
# ==============================================================================
# PHẦN 3: HỆ THỐNG ĐA TÁC TỬ CỘNG TÁC (MULTI-AGENT COLLABORATIVE SYSTEM)
# Khái niệm (Google Cloud [1] & McKinsey [3]): Phân rã bài toán phức tạp cho nhiều tác tử chuyên biệt:
# 1. Planner Agent (Lập kế hoạch phân rã nhiệm vụ)
# 2. Worker Agent (Thực thi công cụ và tính toán chuyên sâu)
# 3. Critic/Reviewer Agent (Thẩm định an toàn, kiểm duyệt đạo đức và chất lượng)
# ==============================================================================

class PlannerAgent:
    def plan(self, task_description: str) -> list:
        return [
            {"id": 1, "step": "Phân tích hồ sơ bệnh nhân và xác định ngưỡng nguy cơ y sinh học."},
            {"id": 2, "step": "Tra cứu phác đồ và đề xuất đơn thuốc tối ưu chi phí từ Long Châu."},
            {"id": 3, "step": "Kiểm duyệt chống chỉ định và phê duyệt phương án điều trị."}
        ]

class WorkerAgent:
    def execute(self, step_info: dict) -> str:
        s_id = step_info["id"]
        if s_id == 1:
            return "Đã xác định: Bệnh nhân Đái tháo đường Tuýp 2, BMI = 29.5 (Tiền béo phì), Glucose = 165 mg/dL."
        elif s_id == 2:
            return "Đã đề xuất: Phác đồ phối hợp Glucophage XR 500mg (1 viên/ngày) + Đi bộ 45 phút/ngày. Chi phí: 135.000 VNĐ/tháng."
        elif s_id == 3:
            return "Đã gửi bản dự thảo phác đồ sang Ban Thẩm định Chuyên môn."

class CriticAgent:
    def review(self, execution_results: list) -> dict:
        # Kiểm tra tính an toàn y khoa
        is_safe = True
        notes = "Phác đồ chuẩn xác theo hướng dẫn ADA 2026. Liều dùng Glucophage XR phù hợp người tiền béo phì. Đạt chuẩn an toàn 100%."
        return {"approved": is_safe, "feedback": notes}

# Điều phối quy trình cộng tác Đa tác tử (Multi-Agent Workflow)
planner = PlannerAgent()
worker = WorkerAgent()
critic = CriticAgent()

print("=== QUY TRÌNH HỢP TÁC HỆ ĐA TÁC TỬ (MULTI-AGENT WORKFLOW) ===")
user_task = "Xây dựng phác đồ điều trị cá nhân hóa cho bệnh nhân Đái tháo đường Tuýp 2 thừa cân"
print(f"Yêu cầu nhiệm vụ: {user_task}\n")

# Bước 1: Planner lập kế hoạch
subtasks = planner.plan(user_task)
print(f"[Planner Agent]: Đã phân rã nhiệm vụ thành {len(subtasks)} bước chiến lược:")
for st in subtasks:
    print(f"  • Bước {st['id']}: {st['step']}")

# Bước 2: Worker thực thi từng bước
print("\n[Worker Agent]: Bắt đầu thực thi các công việc con:")
exec_logs = []
for st in subtasks:
    res = worker.execute(st)
    exec_logs.append(res)
    print(f"  ↳ Hoàn thành bước {st['id']}: {res}")

# Bước 3: Critic thẩm định
review_res = critic.review(exec_logs)
print(f"\n[Critic Agent]: Đánh giá thẩm định chất lượng & An toàn:")
print(f"  ↳ Trạng thái phê duyệt: {'CHẤP THUẬN (APPROVED)' if review_res['approved'] else 'YÊU CẦU LÀM LẠI'}")
print(f"  ↳ Nhận xét chuyên môn: {review_res['feedback']}")
print("\n===> HỆ THỐNG ĐA TÁC TỬ HOÀN THÀNH XUẤT SẮC TOÀN BỘ QUY TRÌNH TỰ CHỦ!")

=== QUY TRÌNH HỢP TÁC HỆ ĐA TÁC TỬ (MULTI-AGENT WORKFLOW) ===
Yêu cầu nhiệm vụ: Xây dựng phác đồ điều trị cá nhân hóa cho bệnh nhân Đái tháo đường Tuýp 2 thừa cân

[Planner Agent]: Đã phân rã nhiệm vụ thành 3 bước chiến lược:
  • Bước 1: Phân tích hồ sơ bệnh nhân và xác định ngưỡng nguy cơ y sinh học.
  • Bước 2: Tra cứu phác đồ và đề xuất đơn thuốc tối ưu chi phí từ Long Châu.
  • Bước 3: Kiểm duyệt chống chỉ định và phê duyệt phương án điều trị.

[Worker Agent]: Bắt đầu thực thi các công việc con:
  ↳ Hoàn thành bước 1: Đã xác định: Bệnh nhân Đái tháo đường Tuýp 2, BMI = 29.5 (Tiền béo phì), Glucose = 165 mg/dL.
  ↳ Hoàn thành bước 2: Đã đề xuất: Phác đồ phối hợp Glucophage XR 500mg (1 viên/ngày) + Đi bộ 45 phút/ngày. Chi phí: 135.000 VNĐ/tháng.
  ↳ Hoàn thành bước 3: Đã gửi bản dự thảo phác đồ sang Ban Thẩm định Chuyên môn.

[Critic Agent]: Đánh giá thẩm định chất lượng & An toàn:
  ↳ Trạng thái phê duyệt: CHẤP THUẬN (APPROVED)
  ↳ Nhận xét chuyên môn: Phác đồ chuẩn xác theo hướng d